In [10]:
import pdfplumber
import nltk
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
import pandas as pd


In [11]:
def extract_cv_text(pdf_path):
    """
    Extract text from a CV PDF using pdfplumber.
    """
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

In [12]:
def predict_personality(cv_text):
    """
    Predict personality traits from CV text.
    - Features: TF-IDF for word choices, sentiment scores for emotional tone.
    - Model: Simple logistic regression trained on synthetic data.
    - Traits: Openness, Conscientiousness, Extraversion, Agreeableness, Emotional Stability.
    """
    # Synthetic training data (in real-world, use a labeled dataset)
    data = pd.DataFrame({
        'text': [
            "Innovative projects with creative solutions and new technologies.", # High Openness
            "Organized tasks, met deadlines, attention to detail in reports.", # High Conscientiousness
            "Led team meetings, networked at conferences, public speaking.", # High Extraversion
            "Collaborated with teams, supported colleagues, volunteer work.", # High Agreeableness
            "Handled high-pressure situations, adapted to changes calmly.", # High Emotional Stability
            "Routine tasks, avoided new ideas, stuck to traditional methods.", # Low Openness
            "Missed deadlines, disorganized work, lacked planning.", # Low Conscientiousness
            "Worked alone, avoided social events, introverted roles.", # Low Extraversion
            "Competitive, disagreed often, focused on individual goals.", # Low Agreeableness
            "Stressed under pressure, emotional reactions to feedback." # Low Emotional Stability
        ],
        'trait': ['Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Emotional Stability',
                  'Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Emotional Stability'],
        'level': ['High', 'High', 'High', 'High', 'High', 'Low', 'Low', 'Low', 'Low', 'Low']
    })
    # Preprocess: Add sentiment as feature
    sia = SentimentIntensityAnalyzer()
    data['sentiment'] = data['text'].apply(lambda x: sia.polarity_scores(x)['compound'])
    # Encode labels (trait_level combo)
    data['label'] = data['trait'] + '_' + data['level']
    le = LabelEncoder()
    y = le.fit_transform(data['label'])
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(data[['text', 'sentiment']], y, test_size=0.2, random_state=42)
    # Pipeline: TF-IDF on text + model
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('clf', LogisticRegression())
    ])
    # Fit on text only (sentiment can be added as extra feature if needed)
    pipeline.fit(X_train['text'], y_train)
    # Predict on input CV
    cv_sentiment = sia.polarity_scores(cv_text)['compound']
    prediction = pipeline.predict([cv_text])
    predicted_trait_level = le.inverse_transform(prediction)[0]
    return predicted_trait_level.replace('_', ' ')

In [14]:
# Replace with your CV PDF path
cv_path = "Roshan_Khadka_Resume 1.pdf"
cv_text = extract_cv_text(cv_path)
print("Extracted CV Text (truncated):")
print(cv_text[:300] + "..." if len(cv_text) > 300 else cv_text)
   
trait = predict_personality(cv_text)
print("\nPredicted Personality Trait:")
print(trait)

Extracted CV Text (truncated):
ROSHAN KHADKA
Birtamode, Nepal | +977 9746879866 | rk4550328@gmail.com | linkedin.com/in/roshankhadka22 |
github.com/lauv22
Education
Gomendra Multiple College — Bachelor of Computer Application (BCA)
Purbanchal University | 2022 – Ongoing | Birtamode, Nepal
Relevant Coursework
Data Structures • Dat...

Predicted Personality Trait:
Openness High
